In [ ]:
# ---------------- Imports ----------------

import os
import json
from itertools import combinations

import yaml
import numpy as np
import pandas as pd

from scipy.stats import spearmanr


pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")



In [ ]:
# ---------------- Inputs ----------------

MODELS = {
    "STSB":       "ibm-rank-30k-samp2/cross-encoder/stsb-roberta-large/20260922t171221-llm-consistency.jsonl", # "Llama 3B"
    "L3":         "ibm-rank-30k-samp2/meta-llama/Llama-3.2-3B-Instruct/20260912t000622-llm-consistency.jsonl", # "Llama 3B"
    "L8":         "ibm-rank-30k-samp2/meta-llama/Llama-3.1-8B-Instruct/20260912t001205-llm-consistency.jsonl", # "Llama 8B"
    "L70":        "ibm-rank-30k-samp2/meta-llama/Llama-3.3-70B-Instruct/20260912t153621-llm-consistency.jsonl", # #"Llama 70
    "Q7":         "ibm-rank-30k-samp2/Qwen/Qwen2.5-7B-Instruct/20260912t002447-llm-consistency.jsonl", # "Qwen 7B":
    "Q14":        "ibm-rank-30k-samp2/Qwen/Qwen2.5-14B-Instruct/20260912t003428-llm-consistency.jsonl", # "Qwen 14B"
    "Q72":        "ibm-rank-30k-samp2/Qwen/Qwen2.5-72B-Instruct/20260913t012204-llm-consistency.jsonl", # #"Qwen 72B
    "M8":         "ibm-rank-30k-samp2/mistralai/Ministral-3-8B-Instruct-2512/20260912t005409-llm-consistency.jsonl", # "Mistral 8
    "M14":        "ibm-rank-30k-samp2/mistralai/Ministral-3-14B-Instruct-2512/20260912t010437-llm-consistency.jsonl", # "Mistral 1
    "M24":        "ibm-rank-30k-samp2/mistralai/Mistral-Small-3.2-24B-Instruct-2506/20260912t150257-llm-consistency.jsonl", # "Mistral 2
    "P4":         "ibm-rank-30k-samp2/microsoft/Phi-4-mini-instruct/20260912t143117-llm-consistency.jsonl", # "Phi-4":  
    "G12":         "ibm-rank-30k-samp2/google/gemma-4-12B-it/20260912t222101-llm-consistency.jsonl", # "gemma-4-12B-it":  
}





In [ ]:
# ---------------- Config ----------------

with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

proj_store = config["paths"]["proj-store"]

input_folder = os.path.join(proj_store, "experiments", "llm-consistency")


OUTPUT_FOLDER = os.path.join(proj_store, "evaluation", "llm-consistency", "correlation")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

OUTPUT_PATH = os.path.join(OUTPUT_FOLDER, f"correlation-results")



In [ ]:
# ---------------- Utilities ----------------

def read_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(
                    f"Invalid JSON in {path}, line {line_number}: {e}"
                )

    return records



In [ ]:
# ---------------- Load model results ----------------

model_dfs = {}

for model_name, input_filename in MODELS.items():
    input_path = os.path.join(input_folder, input_filename)
    df = pd.DataFrame(read_jsonl(input_path))

    required_columns = ["argument_id", "evaluation_topic"]

    missing_columns = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{input_filename} is missing columns: {missing_columns}"
        )

    if "semantic_similarity" in df.columns:
        df["score"] = df["semantic_similarity"]

    elif "label_probs" in df.columns:
        df["score"] = df["label_probs"].apply(
            lambda scores: scores["Yes"]
        )

    else:
        raise ValueError(
            f"{input_filename} contains neither semantic_similarity "
            f"nor label_probs."
        )

    df = df[["argument_id", "evaluation_topic", "score"]].copy()

    if df.duplicated(
        subset=["argument_id", "evaluation_topic"]
    ).any():
        raise ValueError(
            f"{model_name} contains duplicate "
            f"argument_id × evaluation_topic rows."
        )

    model_dfs[model_name] = df

    print(
        f"{model_name}: "
        f"{df['argument_id'].nunique():,} arguments, "
        f"{df['evaluation_topic'].nunique():,} topics, "
        f"{len(df):,} evaluations"
    )



In [ ]:
# ---------------- Validate model coverage ----------------

reference_name = next(iter(model_dfs))
reference_df = model_dfs[reference_name]

reference_pairs = set(zip(reference_df["argument_id"], reference_df["evaluation_topic"]))

for model_name, df in model_dfs.items():
    model_pairs = set(zip(df["argument_id"], df["evaluation_topic"]))

    if model_pairs != reference_pairs:
        missing = len(reference_pairs - model_pairs)
        extra = len(model_pairs - reference_pairs)

        raise ValueError(
            f"{model_name} does not match {reference_name}: "
            f"{missing:,} missing pairs, {extra:,} extra pairs."
        )

print()
print("All models contain the same argument × objective evaluations.")



In [ ]:
# ---------------- Merge scores ----------------

merged_df = None

for model_name, df in model_dfs.items():
    model_df = df.rename(columns={"score": model_name})

    if merged_df is None:
        merged_df = model_df
    else:
        merged_df = merged_df.merge(
            model_df,
            on=["argument_id", "evaluation_topic"],
            how="inner",
            validate="one_to_one",
        )

print()
print(f"Merged evaluations: {len(merged_df):,}")



In [ ]:
# ---------------- Pairwise observer agreement ----------------

model_names = list(model_dfs.keys())

correlation_matrix = pd.DataFrame(
    np.eye(len(model_names)),
    index=model_names,
    columns=model_names,
)

pairwise_rows = []

for model_a, model_b in combinations(model_names, 2):
    argument_correlations = []

    for argument_id, group in merged_df.groupby("argument_id"):
        rho = spearmanr(group[model_a], group[model_b]).statistic

        if not np.isnan(rho):
            argument_correlations.append(rho)

    argument_correlations = np.asarray(argument_correlations, dtype=float)

    mean_rho = argument_correlations.mean()

    correlation_matrix.loc[model_a, model_b] = mean_rho
    correlation_matrix.loc[model_b, model_a] = mean_rho

    pairwise_rows.append({
        "model_a": model_a,
        "model_b": model_b,
        "n_arguments": len(argument_correlations),
        "mean_spearman_rho": mean_rho,
        "median_spearman_rho": np.median(argument_correlations),
        "std_spearman_rho": argument_correlations.std(ddof=1),
    })


pairwise_df = pd.DataFrame(pairwise_rows)



In [ ]:
# ---------------- Display ----------------

print("Mean within-argument Spearman correlation")
print()

display(correlation_matrix)

print()
print("Pairwise details")
print()

display(pairwise_df.sort_values("mean_spearman_rho", ascending=False).reset_index(drop=True))


correlation_matrix.to_csv(f"{OUTPUT_PATH}-correlation-matrix.csv", index=False)
pairwise_df.to_csv(f"{OUTPUT_PATH}-pairwise.csv", index=False)

